# RAG sobre la Constitución del Ecuador con LangChain + Ollama

Este notebook demuestra cómo construir un sistema de **Retrieval Augmented Generation (RAG)** paso a paso usando únicamente herramientas locales y de código abierto.

## 1. Cargar los artículos constitucionales

In [38]:
import pandas as pd
from langchain.schema import Document

CSV_PATH = "constitución.csv"

df = pd.read_csv(CSV_PATH)
df.head(5)

,Titulo,Capítulo,Concordancias,Artículo,Numerales,Artículo Completo
0,TÍTULO I ELEMENTOS CONSTITUTIVOS DEL ESTADO,Capítulo primero Principios fundamentales,Concordancias: CONSTITUCIÓN DE LA REPÚBLICA DE...,Art. 1.- El Ecuador es un Estado constituciona...,,Art. 1.- El Ecuador es un Estado constituciona...
1,TÍTULO I ELEMENTOS CONSTITUTIVOS DEL ESTADO,Capítulo primero Principios fundamentales,Concordancias: CONSTITUCIÓN DE LA REPÚBLICA DE...,"Art. 2.- La bandera, el escudo y el himno naci...",,"Art. 2.- La bandera, el escudo y el himno naci..."
2,TÍTULO I ELEMENTOS CONSTITUTIVOS DEL ESTADO,Capítulo primero Principios fundamentales,Concordancias: CONSTITUCIÓN DE LA REPÚBLICA DE...,Art. 3.- Son deberes primordiales del Estado:,1. Garantizar sin discriminación alguna el efe...,Art. 3.- Son deberes primordiales del Estado: ...
3,TÍTULO I ELEMENTOS CONSTITUTIVOS DEL ESTADO,Capítulo primero Principios fundamentales,Concordancias: CONSTITUCIÓN DE LA REPÚBLICA DE...,Art. 3.- Son deberes primordiales del Estado:,2. Garantizar y defender la soberanía nacional.,Art. 3.- Son deberes primordiales del Estado: ...
4,TÍTULO I ELEMENTOS CONSTITUTIVOS DEL ESTADO,Capítulo primero Principios fundamentales,Concordancias: CONSTITUCIÓN DE LA REPÚBLICA DE...,Art. 3.- Son deberes primordiales del Estado:,3. Fortalecer la unidad nacional en la diversi...,Art. 3.- Son deberes primordiales del Estado: ...


## 2. Chunking por artículo

El proceso de **chunking** se refiere a dividir la fuente de conocimento en fragmentos más pequeños para poder hacer la comparación semántica de manera más sencilla y extraer únicamente el fragmento que necesitamos

In [39]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

MAX_CHARS = 1500

splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_CHARS,
    chunk_overlap=150,
)

## Ejemplo de chunking sin recursive splitter
documentos = []

for _, fila in df.iterrows():
    texto = str(fila["Artículo Completo"]).strip()
    if not texto or texto == "nan":
        continue

    encabezado = str(fila.get("Artículo", "")).strip()[:100]
    metadata = {
        "titulo":   str(fila.get("Titulo", "")).strip(),
        "capitulo": str(fila.get("Capítulo", "")).strip(),
        "articulo": encabezado,
    }
    documentos.append(Document(page_content=texto, metadata=metadata))
max_len = max(len(d.page_content) for d in documentos)
print(f"Total de chunks generados: {len(documentos)}")
print(f"Chunk más largo:           {max_len} caracteres")

## Ejemplo de chunking con recursive splitter
documentos = []

for _, fila in df.iterrows():
    texto = str(fila["Artículo Completo"]).strip()
    if not texto or texto == "nan":
        continue

    encabezado = str(fila.get("Artículo", "")).strip()[:100]
    metadata = {
        "titulo":   str(fila.get("Titulo", "")).strip(),
        "capitulo": str(fila.get("Capítulo", "")).strip(),
        "articulo": encabezado,
    }

    if len(texto) <= MAX_CHARS:
        documentos.append(Document(page_content=texto, metadata=metadata))
    else:
        chunks = splitter.split_text(texto)
        for i, chunk in enumerate(chunks):
            if i > 0:
                chunk = f"[Continuación de {encabezado[:60]}] {chunk}"
            documentos.append(Document(page_content=chunk, metadata=metadata))

max_len = max(len(d.page_content) for d in documentos)
print(f"Total de chunks generados: {len(documentos)}")
print(f"Chunk más largo:           {max_len} caracteres")

Total de chunks generados: 1113
Chunk más largo:           26063 caracteres
Total de chunks generados: 1495
Chunk más largo:           1578 caracteres


## 3. Crear embeddings con Ollama

Un **embedding** es una representación numérica del texto en un espacio vectorial. Textos con significado similar quedan cerca en ese espacio y se utiliza métodos como la similitud de cosenos para determinar que tan similar es su significado

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector = embeddings.embed_query("derecho al trabajo")
print(f"Dimensiones del vector: {len(vector)}")
print(vector)

Dimensiones del vector: 768
[0.04166666, 0.016429, -0.17332466, 0.06328811, -0.009950842, -0.04151539, -0.01824174, -0.014840679, -0.08029907, 0.01820374, -0.01026843, 0.003573356, 0.045263875, -0.039164808, 0.081927516, -0.012897874, 0.013531037, -0.045800876, -0.060514696, -0.008257522, 0.02418667, 0.004729442, -0.017456656, -0.039302297, 0.11605544, 0.0038014152, 0.0037390064, 0.049797945, 0.047040246, -0.025212195, -0.07658021, -0.016964367, 0.05867133, 0.0206463, -0.017562829, -0.0693004, 0.04817833, 0.04873952, 0.009882095, -0.0059712217, -0.007297161, -0.007614393, -0.047075134, -0.028043618, 0.04423031, -0.005793015, 0.017320618, 0.05420769, -0.0021205146, 0.005810094, 0.0011885916, -0.00524702, -0.0343529, -0.013642547, 0.07413093, -0.023373576, -0.004511621, -0.034249272, -0.057996936, -0.056266382, 0.07078043, 0.07530473, -0.012654617, 0.09487759, -0.020682393, -0.019285629, 0.008751117, 0.028616967, 0.0032955075, -0.021404441, 0.033482015, -0.010109346, 0.023925735, 0.03441

## 4. Construir el Vector Store con Chroma

In [5]:
import os
from langchain_chroma import Chroma

CHROMA_PATH = "./chroma_db"

if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):
    print("Cargando vector store existente desde disco...")
    vectorstore = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)
else:
    print(f"Creando vector store con {len(documentos)} chunks...")
    vectorstore = Chroma.from_documents(
        documents=documentos,
        embedding=embeddings,
        persist_directory=CHROMA_PATH,
    )

total = vectorstore._collection.count()
print(f"Vector store listo — {total} chunks indexados")

Cargando vector store existente desde disco...
Vector store listo — 1526 chunks indexados


## 5. Configurar el LLM con Ollama

In [6]:
from langchain_ollama import ChatOllama

## La temperatura determina variación en las respuestas, la mantenemos en 0 porque necesitamos extracción textual
llm = ChatOllama(model="llama3.1:8b", temperature=0)

respuesta = llm.invoke("Prueba de conexión")
print(respuesta.content)

Parece que estás intentando establecer una conexión conmigo. ¿En qué puedo ayudarte hoy?


## 6. El problema de la alucinación

Antes de construir el RAG, demostremos **por qué es necesario**.

In [10]:
pregunta = "¿Qué establece el artículo 45 de la Constitución del Ecuador?"

print("PREGUNTA DIRECTA AL LLM (sin RAG)")
print("=" * 50)
respuesta_sin_rag = llm.invoke(pregunta)
print(respuesta_sin_rag.content)

PREGUNTA DIRECTA AL LLM (sin RAG)
El artículo 45 de la Constitución del Ecuador establece que los ecuatorianos tienen derecho a la libertad de conciencia y religión.


In [11]:
# Lo que realmente dice el Art. 45
art_45 = df[df["Artículo"].str.contains("Art. 45", na=False)]
print("LO QUE DICE REALMENTE EL ARTÍCULO 45:")
print("=" * 50)
for _, fila in art_45.iterrows():
    print(fila["Artículo Completo"])
    print()

LO QUE DICE REALMENTE EL ARTÍCULO 45:
Art. 45.- Las niñas, niños y adolescentes gozarán de los derechos comunes del ser humano, además de los específicos de su edad. El Estado reconocerá y garantizará la vida, incluido el cuidado y protección desde la concepción. Las niñas, niños y adolescentes tienen derecho a la integridad física y psíquica; a su identidad, nombre y ciudadanía; a la salud integral y nutrición; a la educación y cultura, al deporte y recreación; a la seguridad social; a tener una familia y disfrutar de la convivencia familiar y comunitaria; a la participación social; al respeto de su libertad y dignidad; a ser consultados en los asuntos que les afecten; a educarse de manera prioritaria en su idioma y en los contextos culturales propios de sus pueblos y nacionalidades; y a recibir información acerca de sus progenitores o familiares ausentes, salvo que fuera perjudicial para su bienestar. El Estado garantizará su libertad de expresión y asociación, el funcionamiento libr

## 7. MultiQueryRetriever: más cobertura semántica

MultiQueryRetriever hace uso de las capacidades del LLM para reformular la pregunta varias veces para encontrar la respuesta correcta, por ejemplo, preguntamos ¿el agua es un derecho en el Ecuador? puede que no encuentre la respuesta, pero si se reformular como ¿existe el derecho al agua en Ecuador? puede que sí.

In [12]:
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever_base = vectorstore.as_retriever(search_kwargs={"k": 3})

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever_base,
    llm=llm,
)

In [14]:
# Demostración: ver las variaciones que genera el LLM para una pregunta
pregunta_demo = "¿Se reconoce en el Ecuador el derecho humano al agua?"

docs_recuperados = multi_retriever.invoke(pregunta_demo)

print(f"\nDocumentos recuperados en total: {len(docs_recuperados)}")
print("-" * 60)
for doc in docs_recuperados:
    print(f"Artículo: {doc.metadata['articulo'][:70]}")
    print(f"Texto:    {doc.page_content}")
    print("-" * 60)


Documentos recuperados en total: 9
------------------------------------------------------------
Artículo: Art. 17.- El Estado fomentará la pluralidad y la diversidad en la comu
Texto:    Art. 17.- El Estado fomentará la pluralidad y la diversidad en la comunicación, y al efecto: 3. No permitirá el oligopolio o monopolio, directo ni indirecto, de la propiedad de los medios de comunicación y del uso de las frecuencias.
------------------------------------------------------------
Artículo: Art. 196.- La Fiscal o el Fiscal General del Estado reunirá los siguie
Texto:    Art. 196.- La Fiscal o el Fiscal General del Estado reunirá los siguientes requisitos: 1. Ser ecuatoriana o ecuatoriano y estar en goce de los derechos políticos.
------------------------------------------------------------
Artículo: Art. 192.- La Defensora Pública o Defensor Público General reunirá los
Texto:    Art. 192.- La Defensora Pública o Defensor Público General reunirá los siguientes requisitos: 1. Ser ecuatorian

## 8. Construir la cadena RAG mejorada

Combinamos el MultiQueryRetriever con el prompt y la memoria conversacional.

In [15]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

SYSTEM_PROMPT = """Eres un abogado ecuatoriano experto en derecho constitucional.

Reglas estrictas:
- Responde ÚNICAMENTE con base en los artículos constitucionales proporcionados.
- Cita siempre el artículo específico que sustenta tu respuesta (ej: "Según el Art. 33...").
- Para preguntas de sí/no: responde primero Sí o No, luego cita el artículo y transcribe
  la parte relevante del texto constitucional que lo sustenta.
- Cuando el artículo tenga numerales relevantes, menciona cuáles aplican y por qué.
- Explica brevemente el significado práctico para el ciudadano.
- Si la respuesta no está en los artículos proporcionados, di:
  "Esta información no está en los artículos relevantes que encontré."
- Responde en español, de forma clara y accesible para un ciudadano común.
- Tus respuestas deben tener al menos 3 oraciones."""

prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template(
        "Artículos constitucionales relevantes:\n{context}\n\n"
        "Historial de conversación:\n{chat_history}\n\n"
        "Pregunta: {question}"
    ),
])

memoria = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

cadena_rag = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=multi_retriever,
    memory=memoria,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True,
)

C:\Users\Jaime\AppData\Local\Temp\ipykernel_13520\3214887795.py:32: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memoria = ConversationBufferMemory(


## 9. Validación con preguntas de prueba

Ejecutamos las 10 preguntas de validación para medir la calidad del sistema.

In [18]:
def consultar(pregunta: str, mostrar_fuentes: bool = True) -> str:
    resultado = cadena_rag.invoke({"question": pregunta})
    respuesta = resultado["answer"]

    print(f"P: {pregunta}")
    print(f"R: {respuesta}")
    if mostrar_fuentes:
        articulos = {doc.metadata['articulo'][:60] for doc in resultado["source_documents"]}
        print(f"   Fuentes: {' | '.join(articulos)}")

In [19]:
memoria.clear()
consultar("¿Se reconoce en el Ecuador el derecho humano al agua?")

P: ¿Se reconoce en el Ecuador el derecho humano al agua?
R: Sí.

El Art. 12 establece que "El derecho humano al agua es fundamental e irrenunciable". Además, el Art. 318 especifica que "El agua constituye patrimonio nacional estratégico de uso público, inalienable, imprescriptible, inembargable y esencial para la vida" y prohíbe toda forma de privatización del agua.

Significa que el Estado tiene la obligación de garantizar el acceso al agua como un derecho humano fundamental, y no puede ser privatizado ni utilizado para fines lucrativos.
   Fuentes: Art. 17.- El Estado fomentará la pluralidad y la diversidad  | Art. 318.- El agua es patrimonio nacional estratégico de uso | Art. 326.- El derecho al trabajo se sustenta en los siguient | Art. 12.- El derecho humano al agua es fundamental e irrenun | Art. 196.- La Fiscal o el Fiscal General del Estado reunirá  | Art. 41.- Se reconocen los derechos de asilo y refugio, de a | Art. 416.- Las relaciones del Ecuador con la comunidad inter | Ar

In [20]:
memoria.clear()
consultar("¿Quiénes son considerados ecuatorianos por nacimiento?")

P: ¿Quiénes son considerados ecuatorianos por nacimiento?
R: Según el Art. 7, los ecuatorianos por nacimiento son:

1. Las personas nacidas en el Ecuador (Art. 7 numeral 1).
2. Las personas pertenecientes a comunidades, pueblos o nacionalidades reconocidos por el Ecuador con presencia en las zonas de frontera (Art. 7 numeral 3).

Esto significa que cualquier persona que nazca dentro del territorio ecuatoriano es considerada ecuatoriana por nacimiento, independientemente de su origen étnico o nacionalidad. Además, también lo son aquellas personas que pertenecen a comunidades indígenas reconocidas por el Estado con presencia en las zonas de frontera.

Significado práctico: Esto garantiza la ciudadanía ecuatoriana a todas las personas nacidas dentro del país y a aquellos miembros de comunidades indígenas que tienen una conexión histórica y cultural con el territorio.
   Fuentes: Art. 17.- El Estado fomentará la pluralidad y la diversidad  | Art. 9.- Las personas extranjeras que se encuent

In [21]:
memoria.clear()
consultar("¿La justicia en el Ecuador garantiza acceso gratuito y efectivo?")

P: ¿La justicia en el Ecuador garantiza acceso gratuito y efectivo?
R: Sí. 

Según el Art. 75, "Toda persona tiene derecho al acceso gratuito a la justicia y a la tutela efectiva, imparcial y expedita de sus derechos e intereses, con sujeción a los principios de inmediación y celeridad; en ningún caso quedará en indefensión."

El significado práctico para el ciudadano es que puede acceder a la justicia sin tener que pagar costas procesales, lo cual facilita el acceso a la justicia para las personas de escasos recursos.
   Fuentes: Art. 168.- La administración de justicia, en el cumplimiento | Art. 426.- Todas las personas, autoridades e instituciones e | Art. 83.- Son deberes y responsabilidades de las ecuatoriana | Art. 147.- Son atribuciones y deberes de la Presidenta o Pre | Art. 196.- La Fiscal o el Fiscal General del Estado reunirá  | Art. 75.- Toda persona tiene derecho al acceso gratuito a la | Art. 92.- Toda persona, por sus propios derechos o como repr | Art. 436.- La Corte Co

In [22]:
memoria.clear()
consultar("¿La Corte Constitucional es el máximo órgano de control e interpretación constitucional?")

P: ¿La Corte Constitucional es el máximo órgano de control e interpretación constitucional?
R: Sí.

Art. 429.- La Corte Constitucional es el máximo órgano de control, interpretación constitucional y de administración de justicia en esta materia. 

La Corte Constitucional tiene la función de garantizar que las leyes y acciones del Estado se ajusten a la Constitución. Esto significa que es el máximo órgano para resolver conflictos relacionados con la interpretación de la Constitución y aplicarla en los casos correspondientes.

Significado práctico: La Corte Constitucional es una institución clave para proteger los derechos fundamentales de los ciudadanos y garantizar que el Estado actúe dentro de los límites establecidos por la Constitución.
   Fuentes: Art. 83.- Son deberes y responsabilidades de las ecuatoriana | Art. 134.- La iniciativa para presentar proyectos de ley cor | Art. 379.- Son parte del patrimonio cultural tangible e inta | Art. 149.- Quien ejerza la Vicepresidencia de la 

In [23]:
memoria.clear()
consultar("¿El Ecuador reconoce la soberanía alimentaria como objetivo estratégico del Estado?")

P: ¿El Ecuador reconoce la soberanía alimentaria como objetivo estratégico del Estado?
R: Sí. 

Según el Art. 281, numeral 1, "La soberanía alimentaria constituye un objetivo estratégico y una obligación del Estado para garantizar que las personas, comunidades, pueblos y nacionalidades alcancen la autosuficiencia de alimentos sanos y culturalmente apropiado de forma permanente."

El significado práctico es que el Estado debe priorizar la producción y distribución de alimentos locales, sostenibles y culturamente relevantes para garantizar la seguridad alimentaria de las personas y comunidades.
   Fuentes: Art. 410.- El Estado brindará a los agricultores y a las com | Art. 147.- Son atribuciones y deberes de la Presidenta o Pre | Art. 284.- La política económica tendrá los siguientes objet | Art. 261.- El Estado central tendrá competencias exclusivas  | Art. 281.- La soberanía alimentaria constituye un objetivo e


In [33]:
memoria.clear()
consultar("¿Se puede sentenciar pena de muerte en Ecuador?")

P: ¿Se puede sentenciar pena de muerte en Ecuador?
R: No. Según el Art. 66, No habrá pena de muerte.

Significa que la Constitución ecuatoriana prohíbe expresamente la pena de muerte como sanción penal. Esto garantiza la vida y la integridad física de las personas, protegiendo su derecho a la inviolabilidad de la vida.
   Fuentes: Art. 17.- El Estado fomentará la pluralidad y la diversidad  | Art. 83.- Son deberes y responsabilidades de las ecuatoriana | Art. 196.- La Fiscal o el Fiscal General del Estado reunirá  | Art. 79.- En ningún caso se concederá la extradición de una  | Art. 148.- "La Presidenta o Presidente de la República podrá | Art. 192.- La Defensora Pública o Defensor Público General r | Art. 77.- En todo proceso penal en que se haya privado de la | Art. 66.- Se reconoce y garantizará a las personas:


In [25]:
memoria.clear()
consultar("¿El derecho a la identidad en Ecuador incluye el nombre y el apellido?")

P: ¿El derecho a la identidad en Ecuador incluye el nombre y el apellido?
R: Sí. 
Según el Art. 66, el derecho a la identidad personal y colectiva incluye "tener nombre y apellido, debidamente registrados y libremente escogidos".

Este artículo garantiza que las personas tengan la libertad de elegir su propio nombre y apellido, lo cual es un aspecto fundamental de la identidad personal. Esto significa que los ciudadanos ecuatorianos tienen derecho a decidir cómo se les conoce y cómo se registra su identidad en documentos oficiales.

En la práctica, esto implica que las personas pueden cambiar su nombre o apellido si lo desean, siempre y cuando lo hagan de manera voluntaria y sin presión. Esto puede ser importante para aquellos que han cambiado su nombre debido a razones personales, culturales o religiosas.

Es importante destacar que el artículo 66 también menciona la importancia de conservar, desarrollar y fortalecer las características materiales e inmateriales de la identidad, lo cu

In [35]:
memoria.clear()
consultar("¿El matrimonio en Ecuador se reconoce solo para personas heterosexuales?")

P: ¿El matrimonio en Ecuador se reconoce solo para personas heterosexuales?
R: Sí.
Art. 67.- Se reconoce la familia en sus diversos tipos. El Estado la protegerá como núcleo fundamental de la sociedad y garantizará condiciones que favorezcan integralmente la consecución de sus fines. Estas se constituirán por vínculos jurídicos o de hecho y se basarán en la igualdad de derechos y oportunidades de sus integrantes. El matrimonio es la unión entre hombre y mujer, se fundará en el libre consentimiento de las personas contrayentes y en la igualdad de sus derechos, obligaciones y capacidad legal.

Significa que, según la Constitución ecuatoriana, el matrimonio tradicionalmente reconocido es entre una persona hombre y otra persona mujer. Sin embargo, también reconoce otras formas de familia como las uniones de hecho (Art. 68).
   Fuentes: Art. 17.- El Estado fomentará la pluralidad y la diversidad  | Art. 68.- La unión estable y monogámica entre dos personas l | Art. 67.- Se reconoce la famil

In [36]:
memoria.clear()
consultar("¿Un extranjero adoptado por ecuatorianos puede obtener la nacionalidad por naturalización?")

P: ¿Un extranjero adoptado por ecuatorianos puede obtener la nacionalidad por naturalización?
R: Sí, según el Art. 8 numeral 2, las extranjeras menores de edad adoptadas por una ecuatoriana o ecuatoriano pueden obtener la nacionalidad ecuatoriana por naturalización.

"Son ecuatorianas y ecuatorianos por naturalización las siguientes personas: 2. Las extranjeras menores de edad adoptadas por una ecuatoriana o ecutoriano, que conservarán la nacionalidad ecuatoriana mientras no expresen voluntad contraria."

Significa prácticamente que un menor de edad adoptado por ecuatorianos puede obtener la nacionalidad ecuatoriana sin necesidad de cumplir con los requisitos habituales para la naturalización.
   Fuentes: Art. 61.- Las ecuatorianas y ecuatorianos gozan de los sigui | Art. 379.- Son parte del patrimonio cultural tangible e inta | Art. 119.- Para ser asambleísta se requerirá tener nacionali | Art. 9.- Las personas extranjeras que se encuentren en el te | Art. 63.- Las ecuatorianas y ecua

In [37]:
memoria.clear()
consultar("¿Es la naturaleza sujeto de derechos en Ecuador?")

P: ¿Es la naturaleza sujeto de derechos en Ecuador?
R: Sí, según el Art. 10 de la Constitución ecuatoriana, "La naturaleza será sujeto de aquellos derechos que le reconozca la Constitución".

"Las personas, comunidades, pueblos, nacionalidades y colectivos son titulares y gozarán de los derechos garantizados en la Constitución y en los instrumentos internacionales. La naturaleza será sujeto de aquellos derechos que le reconozca la Constitución."

Significa que la naturaleza tiene reconocidos derechos específicos en la Constitución, como el derecho a la restauración (Art. 72) y el derecho a ser protegida por el Estado (Art. 397). Esto significa que los ciudadanos pueden exigir al Estado que se respeten estos derechos de la naturaleza y que se tomen medidas para protegerla.
   Fuentes: Art. 17.- El Estado fomentará la pluralidad y la diversidad  | Art. 326.- El derecho al trabajo se sustenta en los siguient | Art. 71.- La naturaleza o Pacha Mama, donde se reproduce y r | Art. 196.- La Fi

## 10. Demo: memoria conversacional

In [29]:
memoria.clear()
consultar("¿Cuáles son los derechos de los trabajadores en Ecuador?")
consultar("¿Y qué pasa si el empleador no cumple esas garantías?", mostrar_fuentes=False)

P: ¿Cuáles son los derechos de los trabajadores en Ecuador?
R: Según el Art. 326, el derecho al trabajo se sustenta en varios principios importantes para los trabajadores en Ecuador. Entre ellos se encuentran:

* El Estado estimulará la creación de organizaciones de las trabajadoras y trabajadores (principio 8).
* Los derechos laborales son irrenunciables e intangibles, por lo que toda estipulación en contrario será nula (principio 2).
* Se prohíbe toda forma de precarización, como la intermediación laboral y la tercerización en las actividades propias y habituales de la empresa o persona empleadora.
* El Estado impulsará la formación y capacitación para mejorar el acceso y calidad del empleo.

Estos principios buscan proteger los derechos de los trabajadores y promover una relación laboral más justa y equitativa. Significa que los trabajadores en Ecuador tienen derecho a organizarse, a tener derechos laborales irrenunciables e intangibles, y a estar protegidos contra formas de precari

In [30]:
# Ver el historial acumulado en memoria
print("HISTORIAL EN MEMORIA:")
for msg in memoria.chat_memory.messages:
    rol = "Usuario" if msg.type == "human" else "Asistente"
    print(f"[{rol}]: {msg.content[:100]}...")
    print()

HISTORIAL EN MEMORIA:
[Usuario]: ¿Cuáles son los derechos de los trabajadores en Ecuador?...

[Asistente]: Según el Art. 326, el derecho al trabajo se sustenta en varios principios importantes para los traba...

[Usuario]: ¿Y qué pasa si el empleador no cumple esas garantías?...

[Asistente]: Sí.
Según el Art. 327, "El incumplimiento de obligaciones, el fraude, la simulación, y el enriquecim...

